In [ ]:
import pandas as pd
from tensorflow import keras
from keras import Sequential, Input
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# --- 1. 데이터 로드 및 전처리 ---
df = pd.read_csv("train.csv")
# 범주형 변수를 원-핫 인코딩
df = pd.get_dummies(df)

# 상관관계 분석 및 특성 선택 (상위 5개 특성 사용)
df_corr = df.corr()
df_corr_sort = df_corr.sort_values("Monthly_Revenue", ascending=False)
# Monthly_Revenue 자신을 제외한 상위 5개 특성 선택
top5 = df_corr_sort["Monthly_Revenue"].head(6) 
cols = list(top5.index)
cols_train = cols[1:] # 선택된 예측 특성 (예: 5개)

y = df["Monthly_Revenue"].values
x = df[cols_train]

# --- 2. 특성 스케일링 (StandardScaler) ---
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# 훈련/테스트 데이터 분할
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42) # 재현성을 위해 random_state 추가

# --- 3. 모델 구조 개선 (더 깊고 넓은 구조 + Dropout) ---
model = Sequential()
model.add(Input(shape=(x_train.shape[1],)))
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3)) # 과적합 방지
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.3)) # 과적합 방지
model.add(Dense(128, activation="relu"))
model.add(Dense(1)) # 회귀 문제의 출력층 (선형 활성화)

model.summary()

# --- 4. 모델 컴파일 및 학습 설정 ---
model.compile(optimizer="adam", loss="mean_squared_error")

# EarlyStopping의 patience를 늘려 더 오래 학습할 기회 제공
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=50) 
modelpath= "./data/model/best_model_v2.keras" # 모델 파일명 변경
checkpointer= ModelCheckpoint(filepath=modelpath, monitor="val_loss",verbose=0, save_best_only=True)

# 모델 학습
print("\n🔄 모델 학습 시작...")
history = model.fit(x_train, y_train,
validation_split=0.25, epochs=2000, batch_size=64, # Batch size 64로 변경
callbacks=[early_stopping_callback, checkpointer], verbose=0)
print("✅ 모델 학습 완료 (Early Stopping 적용)")


# --- 5. 테스트 세트 예측 및 평가 ---
# 저장된 최적 모델 로드 (혹시 학습이 중단된 경우를 대비)
best_model = keras.models.load_model(modelpath)

# 테스트 세트 예측
y_pred_test = best_model.predict(x_test).flatten()

# 결정계수 계산
r2 = r2_score(y_test, y_pred_test)

print(f"---")
print(f" 테스트 데이터의 결정계수 (R²): {r2:.4f}")
print(f"---")

# --- 6. 최종 test.csv 예측 및 파일 생성 ---
test_df = pd.read_csv("test.csv")

# 훈련 시 선택된 특성만 사용
x_test_final = test_df[cols_train]
# 훈련 시 사용한 동일한 scaler로 변환 (필수)
x_test_final_scaled = scaler.transform(x_test_final) 

# 최종 예측을 위해 최적 모델 로드
final_model = keras.models.load_model(modelpath) 

y_pred = final_model.predict(x_test_final_scaled)

# 제출 파일 생성
df_sub = pd.read_csv("submission.csv")
# Monthly_Revenue 열에 예측값 대입
df_sub["Monthly_Revenue"] = y_pred

# 파일 저장
df_sub.to_csv("new_submission.csv", index=False)

print(" 최종 예측 완료! 'new_submission.csv' 파일이 생성되었습니다.")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,817 (261.00 KB)

 Trainable params: 66,817 (261.00 KB)

 Non-trainable params: 0 (0.00 B)


🔄 모델 학습 시작...
✅ 모델 학습 완료 (Early Stopping 적용)
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step 
---
⭐ 테스트 데이터의 결정계수 (R²): 0.6217
---


KeyError: "['Cuisine_Type_Japanese'] not in index"

In [ ]:
import pandas as pd
from tensorflow import keras
from keras import Sequential, Input
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# --- 1. 데이터 로드 및 전처리 ---
df = pd.read_csv("train.csv")
# 범주형 변수를 원-핫 인코딩
df = pd.get_dummies(df)

# 상관관계 분석 및 특성 선택 (상위 5개 특성 사용)
df_corr = df.corr()
df_corr_sort = df_corr.sort_values("Monthly_Revenue", ascending=False)
# Monthly_Revenue 자신을 제외한 상위 5개 특성 선택
top5 = df_corr_sort["Monthly_Revenue"].head(6) 
cols = list(top5.index)
cols_train = cols[1:] # 선택된 예측 특성

y = df["Monthly_Revenue"].values
x = df[cols_train]

# --- 2. 특성 스케일링 (StandardScaler) ---
# 스케일러 정의 및 훈련 데이터에 fit_transform 적용
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# 훈련/테스트 데이터 분할
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

# --- 3. 모델 구조 개선 (더 깊고 넓은 구조 + Dropout) ---
model = Sequential()
model.add(Input(shape=(x_train.shape[1],)))
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3)) # 과적합 방지
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.3)) # 과적합 방지
model.add(Dense(128, activation="relu"))
model.add(Dense(1)) # 회귀 문제의 출력층 (선형 활성화)

model.summary()

# --- 4. 모델 컴파일 및 학습 설정 ---
model.compile(optimizer="adam", loss="mean_squared_error")

# EarlyStopping의 patience를 50으로 늘림
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=50) 
modelpath= "./data/model/best_model_v2.keras"
checkpointer= ModelCheckpoint(filepath=modelpath, monitor="val_loss",verbose=0, save_best_only=True)

# 모델 학습
print("\n 모델 학습 시작")
history = model.fit(x_train, y_train,
validation_split=0.25, epochs=2000, batch_size=64, # Batch size 64로 변경
callbacks=[early_stopping_callback, checkpointer], verbose=0)
print(" 모델 학습 완료 (Early Stopping 적용)")


# --- 5. 테스트 세트 예측 및 평가 ---
best_model = keras.models.load_model(modelpath)

# 테스트 세트 예측
y_pred_test = best_model.predict(x_test).flatten()

# 결정계수 계산
r2 = r2_score(y_test, y_pred_test)

print(f"---")
print(f" 테스트 데이터의 결정계수 (R²): {r2:.4f}")
print(f"---")

# --- 6. 최종 test.csv 예측 및 파일 생성 (KeyError 해결 로직 포함) ---
test_df = pd.read_csv("test.csv")

# 1. 테스트 데이터에 원-핫 인코딩 적용
test_df_dummies = pd.get_dummies(test_df)

# 2. 훈련 데이터에만 있고 테스트 데이터에 없는 특성을 찾아서 0으로 채워 넣음 (KeyError 해결)
missing_cols = set(cols_train) - set(test_df_dummies.columns)

print(f" 테스트 데이터에 누락된 특성: {missing_cols}. 이들을 0으로 채웁니다.")

for c in missing_cols:
    test_df_dummies[c] = 0 # 누락된 특성 컬럼을 0으로 채워 추가

# 3. 훈련 데이터와 동일한 순서로 특성을 선택
x_test_final = test_df_dummies[cols_train] 

# 4. 훈련 시 사용한 동일한 scaler로 변환 (필수)
x_test_final_scaled = scaler.transform(x_test_final) 

# 최종 예측을 위해 최적 모델 로드
final_model = keras.models.load_model(modelpath) 

y_pred = final_model.predict(x_test_final_scaled)

# 제출 파일 생성
df_sub = pd.read_csv("submission.csv")
# Monthly_Revenue 열에 예측값 대입
df_sub["Monthly_Revenue"] = y_pred.flatten()

# 파일 저장
df_sub.to_csv("new_submission.csv", index=False)

print(" 최종 예측 완료! 'new_submission.csv' 파일이 생성되었습니다.")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,817 (261.00 KB)

 Trainable params: 66,817 (261.00 KB)

 Non-trainable params: 0 (0.00 B)


🔄 모델 학습 시작...
✅ 모델 학습 완료 (Early Stopping 적용)
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step 
---
⭐ 테스트 데이터의 결정계수 (R²): 0.6078
---
⚠️ 테스트 데이터에 누락된 특성: set(). 이들을 0으로 채웁니다.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
✅ 최종 예측 완료! 'new_submission.csv' 파일이 생성되었습니다.


In [ ]:
import pandas as pd
from tensorflow import keras
from keras import Sequential, Input
from keras.layers import Dense, BatchNormalization
from keras.regularizers import l2
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ============================
# 1. 데이터 로드 및 전처리
# ============================
df = pd.read_csv("train.csv")
df = pd.get_dummies(df)

df_corr = df.corr()
df_corr_sort = df_corr.sort_values("Monthly_Revenue", ascending=False)

top5 = df_corr_sort["Monthly_Revenue"].head(6)
cols = list(top5.index)
cols_train = cols[1:]

y = df["Monthly_Revenue"].values
x = df[cols_train]

# 스케일링
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=42
)

# ============================
# 2. 최적화된 ANN 모델 정의
# ============================
model = Sequential([
    Input(shape=(x_train.shape[1],)),

    Dense(128, activation="relu", kernel_regularizer=l2(0.001)),
    BatchNormalization(),

    Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    BatchNormalization(),

    Dense(32, activation="relu"),
    BatchNormalization(),

    Dense(1)
])

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.0007, weight_decay=0.001),
    loss="mse"
)

model.summary()

# ============================
# 3. 콜백 설정
# ============================
early_stop = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)

modelpath = "./data/model/best_model_v3.keras"
checkpointer = ModelCheckpoint(modelpath, monitor="val_loss", save_best_only=True)

# ============================
# 4. 모델 학습
# ============================
print("\n 모델 학습 시작")
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=600,
    batch_size=32,
    callbacks=[early_stop, checkpointer],
    verbose=0
)
print("모델 학습 완료")

# ============================
# 5. 평가
# ============================
best_model = keras.models.load_model(modelpath)

y_pred_test = best_model.predict(x_test).flatten()
r2 = r2_score(y_test, y_pred_test)

print(f"---")
print(f" 테스트 데이터의 결정계수 (R²): {r2:.4f}")
print(f"---")

# ============================
# 6. test.csv 예측
# ============================
test_df = pd.read_csv("test.csv")
test_df = pd.get_dummies(test_df)

missing = set(cols_train) - set(test_df.columns)
for c in missing:
    test_df[c] = 0

x_test_final = test_df[cols_train]
x_test_final_scaled = scaler.transform(x_test_final)

y_pred = best_model.predict(x_test_final_scaled)

df_sub = pd.read_csv("submission.csv")
df_sub["Monthly_Revenue"] = y_pred.flatten()
df_sub.to_csv("new_submission.csv", index=False)

print(" 'new_submission.csv' 파일 생성 완료")


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,033 (47.00 KB)

 Trainable params: 11,585 (45.25 KB)

 Non-trainable params: 448 (1.75 KB)


🔄 모델 학습 시작...
✅ 모델 학습 완료
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step 
---
⭐ 테스트 데이터의 결정계수 (R²): 0.4630
---
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
✅ 'new_submission.csv' 파일 생성 완료
